# Semin et al. (2026) datasets — loading & exploration

Companion data for **"Strategies for Span Labeling with Large Language Models"**
([arXiv:2601.16946](https://arxiv.org/abs/2601.16946), Semin, Dušek & Kasner),
the concurrent work we compare against.

Their repo ([semindan/span_labeling](https://github.com/semindan/span_labeling))
**gitignores `data/`**, so the JSON files are not published — but the parsers are,
and the raw sources are public. `scripts/data/prepare_semin_data.py` rebuilds them:

```bash
python scripts/data/prepare_semin_data.py
```

| task | source | examples | status |
|---|---|---|---|
| NER | UniversalNER v1, 18 test sets | 7,523 | reproduced exactly |
| MT error spans | WMT24 ESA via `llm-span-annotators/span-annotation` | 867 | reproduced exactly |
| GEC | MultiGEC / Write & Improve 2024 | 504 | licence-gated, manual step |
| CPL | their synthetic generator | 1,000 | regenerable, not identical |

**Why we want these:** their tagging family (`xml`) is the closest analogue of our
markup approach, and they could *not* apply constrained decoding to it — every
`*_constrained` run in their results is a JSON method. Our trie processor closes
exactly that gap, so UniversalNER is the shared benchmark for the claim.

In [1]:
import json
import importlib.util
from collections import Counter
from pathlib import Path

import pandas as pd

REPO = Path.cwd()
while not (REPO / "src").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent

DATA = REPO / "data" / "semin"
THEIRS = DATA / "raw" / "span_labeling"

print("repo :", REPO)
print("data :", DATA, "->", "OK" if DATA.is_dir() else "MISSING - run scripts/data/prepare_semin_data.py")

repo : /home/stulcrad/master_thesis
data : /home/stulcrad/master_thesis/data/semin -> OK


In [2]:
def load_task(name: str) -> dict[str, list[dict]]:
    """Load every JSON file for a task into {dataset_name: [examples]}."""
    directory = DATA / name
    if not directory.is_dir():
        print(f"missing {directory}")
        return {}
    return {p.stem: json.loads(p.read_text()) for p in sorted(directory.glob("*.json"))}


ner = load_task("universal_ner")
wmt = load_task("wmt")
multigec = load_task("multigec")   # empty unless you did the licence-gated step

print(f"ner      {len(ner):>3} files, {sum(map(len, ner.values())):>6,} examples  (paper: 7,523)")
print(f"wmt      {len(wmt):>3} files, {sum(map(len, wmt.values())):>6,} examples  (paper:   867)")
print(f"multigec {len(multigec):>3} files, {sum(map(len, multigec.values())):>6,} examples  (paper:   504)")

missing /home/stulcrad/master_thesis/data/semin/multigec
ner       18 files,  7,523 examples  (paper: 7,523)
wmt       24 files,    867 examples  (paper:   867)
multigec   0 files,      0 examples  (paper:   504)


## 1. Schema

Every task uses one flat schema, so the same code reads all of them:

```python
{"text": str,
 "spans": [{"text": str, "label": str, "start": int, "end": int}],
 ...task-specific extras}      # wmt adds "source"; multigec adds "correction"
```

`start`/`end` are **character** offsets into `text`.

In [3]:
example = ner["uner_en_ewt"][0]
print(json.dumps(example, indent=2, ensure_ascii=False))

{
  "text": "What is this Miramar ?",
  "spans": [
    {
      "text": "Miramar",
      "label": "LOC",
      "start": 13,
      "end": 20
    }
  ]
}


## 2. Per-dataset overview

In [4]:
def describe(task_name: str, datasets: dict[str, list[dict]]) -> pd.DataFrame:
    rows = []
    for name, examples in datasets.items():
        n_spans = sum(len(e["spans"]) for e in examples)
        lengths = [len(e["text"]) for e in examples]
        rows.append({
            "task": task_name,
            "dataset": name,
            "examples": len(examples),
            "spans": n_spans,
            "spans/ex": round(n_spans / max(len(examples), 1), 2),
            "chars_mean": round(sum(lengths) / max(len(lengths), 1)),
            "chars_max": max(lengths, default=0),
            "labels": ", ".join(sorted({s["label"] for e in examples for s in e["spans"]})),
        })
    return pd.DataFrame(rows)


overview = pd.concat([describe("ner", ner), describe("wmt", wmt), describe("multigec", multigec)],
                     ignore_index=True)
overview

,task,dataset,examples,spans,spans/ex,chars_mean,chars_max,labels
0,ner,uner_ceb_gja,41,49,1.20,38,71,"LOC, ORG, PER"
1,ner,uner_da_ddt,250,446,1.78,118,380,"LOC, ORG, PER"
2,ner,uner_de_pud,585,1039,1.78,139,307,"LOC, ORG, PER"
3,ner,uner_en_ewt,709,1088,1.53,78,405,"LOC, ORG, PER"
4,ner,uner_en_pud,585,1075,1.84,123,327,"LOC, ORG, PER"
5,ner,uner_hr_set,660,1403,2.13,145,377,"LOC, ORG, PER"
6,ner,uner_pt_bosque,623,1215,1.95,144,519,"LOC, ORG, PER"
7,ner,uner_pt_pud,592,1099,1.86,132,322,"LOC, ORG, PER"
8,ner,uner_ru_pud,570,1036,1.82,129,325,"LOC, ORG, PER"
9,ner,uner_sk_snk,557,915,1.64,85,260,"LOC, ORG, PER"


In [5]:
# Label distribution across all of UniversalNER.
# Note: PER / ORG / LOC only -- there is no MISC, unlike CoNLL-2003.
label_counts = Counter(s["label"] for exs in ner.values() for e in exs for s in e["spans"])
pd.Series(label_counts).sort_values(ascending=False).to_frame("count")

,count
LOC,5712
PER,5280
ORG,3170


## 3. Reading spans in context

Renders gold spans inline so you can eyeball annotation style before trusting the numbers.

In [6]:
def render(example: dict, open_tag="[", close_tag="]") -> str:
    """Inline-annotate text with its gold spans."""
    text, out, cursor = example["text"], [], 0
    for span in sorted(example["spans"], key=lambda s: (s["start"], s["end"])):
        out.append(text[cursor:span["start"]])
        out.append(f'{open_tag}{text[span["start"]:span["end"]]}|{span["label"]}{close_tag}')
        cursor = span["end"]
    out.append(text[cursor:])
    return "".join(out)


for name in ["uner_en_ewt", "uner_zh_pud", "uner_ru_pud"]:
    print(f"--- {name} ---")
    for ex in ner[name][:3]:
        print(" ", render(ex))
    print()

--- uner_en_ewt ---
  What is this [Miramar|LOC] ?
  It is a place in [Argentina|LOC] lol
  " In [Argentina|LOC] , beef is revered , respected , and praised .

--- uner_zh_pud ---
  " 雖然 [美國|LOC] 的 許多 數字化 轉型 都是 史無前例 的 ， 但 權力 的 和平 轉移 卻 存在 先例 ， ” [奧巴馬|PER] 的 特別 助理 [科瑞 · 舒爾曼|PER] 在 周 一 發布 的 博客 中 寫道 。
  對於 通過 社交 媒體 來 跟踪 [國會山|LOC] 任職 變遷 的 人 而 言 ， 這 次 與 以往 有 所 不同 。
  但是 ， 他 在 過去 的 一 次 進行 移民 削減 相關 演說 的 休息 環節 ， 作 為 [共和黨|ORG] 候選人 ， 他 曾 宣稱 ， 作 為 總統 ， 基 於 政績 的 考量 他 將 允許 “ 超 大量 ” 合法 移民 。

--- uner_ru_pud ---
  « Если передача цифровых технологий сегодня в [США|LOC] происходит впервые , то о мирной передаче власти такого не скажешь » , – написала [Кори Шульман|PER] , специальный помощник президента [Обамы|PER] в своем блоге в понедельник .
  Для тех , кто следит за передачей всех материалов , появившихся в социальных сетях о [Конгрессе|ORG] , это будет происходить несколько по-другому .
  Но в отступлении от риторики прошлого о сокращении иммиграции кандидат [Республиканской партии|ORG] заявил , чт

## 4. Does it drop into our existing NER pipeline?

Their `UniversalNERParser` builds `text` as `" ".join(tokens)` — **the same
whitespace-joined convention `evaluationNER_cons_gen.py` already uses for CoNLL**.
So `tokens = text.split(" ")` recovers the original tokenization exactly, and our
`spans_to_bio_tags` consumes their span dicts unchanged (same `start`/`end`/`label` keys).

The cell below verifies that on all 7,523 examples: every gold span must land on
token boundaries, and no span may fail to align.

In [7]:
import sys
sys.path.insert(0, str(REPO / "src"))
from utils.utils_functions import spans_to_bio_tags, build_token_char_spans

UNER_LABELS = {"PER", "ORG", "LOC"}

total = token_aligned = unaligned = 0
for examples in ner.values():
    for ex in examples:
        tokens = ex["text"].split(" ")
        starts = {a for a, _ in build_token_char_spans(tokens)}
        ends = {b for _, b in build_token_char_spans(tokens)}
        if all(s["start"] in starts and s["end"] in ends for s in ex["spans"]):
            token_aligned += 1
        _, n_unaligned = spans_to_bio_tags(tokens, ex["spans"], UNER_LABELS)
        unaligned += n_unaligned
        total += 1

print(f"examples                 {total:,}")
print(f"gold spans on token bnds {token_aligned:,} / {total:,}  ({100 * token_aligned / total:.2f}%)")
print(f"spans our converter drops {unaligned}")
assert token_aligned == total and unaligned == 0
print("\n-> UniversalNER plugs into the CoNLL pipeline unchanged; only the label set differs.")

examples                 7,523
gold spans on token bnds 7,523 / 7,523  (100.00%)
spans our converter drops 0

-> UniversalNER plugs into the CoNLL pipeline unchanged; only the label set differs.


In [8]:
# What a converted example looks like on our side.
ex = ner["uner_en_ewt"][0]
tokens = ex["text"].split(" ")
tags, _ = spans_to_bio_tags(tokens, ex["spans"], UNER_LABELS)
pd.DataFrame({"token": tokens, "gold_tag": tags}).T

,0,1,2,3,4
token,What,is,this,Miramar,?
gold_tag,O,O,O,B-LOC,O


## 5. Their metric

They do **not** use seqeval. Their metric is a character-overlap F1, pooled over a
whole dataset (micro), in two variants:

- **hard** — an overlapping character only counts if the labels also match
- **soft** — label-agnostic, pure span localisation

Zero-length spans (GEC insertion points) get weight 1. This matters: our seqeval
strict-IOB2 F1 is a *stricter, different* number, so **to compare against them we
must report their metric too.** Loaded straight from their checkout so there is no
reimplementation drift.

In [9]:
spec = importlib.util.spec_from_file_location(
    "semin_metrics", THEIRS / "span_labeling" / "metrics.py")
semin_metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(semin_metrics)

gold = ner["uner_en_ewt"][0]["spans"]

print("perfect prediction  ", semin_metrics.compute_overlap_f1(gold, gold))
print("right span, wrong label:")
wrong = [{**s, "label": "ORG" if s["label"] != "ORG" else "PER"} for s in gold]
print("  hard", semin_metrics.compute_overlap_f1(wrong, gold, hard_matching=True))
print("  soft", semin_metrics.compute_overlap_f1(wrong, gold, hard_matching=False))
print("empty prediction    ", semin_metrics.compute_overlap_f1([], gold))

perfect prediction   {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'overlap_chars': 7, 'predicted_chars': 7, 'gold_chars': 7}
right span, wrong label:
  hard {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'overlap_chars': 0, 'predicted_chars': 7, 'gold_chars': 7}
  soft {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'overlap_chars': 7, 'predicted_chars': 7, 'gold_chars': 7}
empty prediction     {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'overlap_chars': 0, 'predicted_chars': 0, 'gold_chars': 7}


In [10]:
def dataset_f1(predictions: list[list[dict]], golds: list[list[dict]], hard=True) -> dict:
    """Pool character counts across a dataset, then compute one micro F1.

    This is their aggregation: counts are summed first, F1 computed once.
    Do NOT average per-example F1 -- that gives a different number.
    """
    overlap = predicted = gold_chars = 0
    for pred, gold in zip(predictions, golds):
        counts = semin_metrics.compute_overlap_counts(pred, gold, hard_matching=hard)
        overlap += counts["overlap_chars"]
        predicted += counts["predicted_chars"]
        gold_chars += counts["gold_chars"]
    return semin_metrics.f1_from_counts(overlap, predicted, gold_chars)


golds = [e["spans"] for e in ner["uner_en_ewt"]]
print("oracle on en_ewt:", dataset_f1(golds, golds))

oracle on en_ewt: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


## 6. Their published baselines

`results/results.csv` in their repo **is** committed — 16,875 runs, the full grid
behind the paper. We can cite their numbers directly instead of re-running their models.

In [11]:
results = pd.read_csv(THEIRS / "results" / "results.csv")
print(results.shape)
print("\nmodels:")
print(results["model"].value_counts())
print("\nmethods:")
print(results["method_name"].value_counts())

(16875, 41)

models:
model
Qwen/Qwen3-8B                                4230
google/gemma-4-31B-it                        2970
meta-llama/Llama-3.3-70B-Instruct            2475
mistralai/Mistral-Small-24B-Instruct-2501    2475
gpt-5-mini                                   1575
NousResearch/Hermes-3-Llama-3.1-8B           1575
mistralai/Mistral-7B-Instruct-v0.3           1575
Name: count, dtype: int64

methods:
method_name
index_enriched                            1845
index                                     1845
xml                                       1845
json_occurrence_structured                1845
json_occurrence                           1845
json_structured                           1845
json                                      1845
json_occurrence_structured_constrained     990
json_occurrence_constrained                990
json_structured_constrained                990
json_constrained                           990
Name: count, dtype: int64


In [12]:
# The key observation for our paper: constrained decoding was only ever applied to
# the JSON families. There is no constrained tagging run -- that is the gap we fill.
pd.crosstab(results["method_name"], results["constrained"])

constrained,False,True
method_name,,
index,1845,0
index_enriched,1845,0
json,1845,0
json_constrained,0,990
json_occurrence,1845,0
json_occurrence_constrained,0,990
json_occurrence_structured,1845,0
json_occurrence_structured_constrained,0,990
json_structured,1845,0


In [13]:
# NER baselines: mean over the 18 UNER datasets and 5 seeds, per model x method.
ner_results = results[results["dataset_type"] == "ner"]
baseline = (ner_results
            .groupby(["model", "method_name"])[["hard_f1", "soft_f1"]]
            .mean()
            .round(4)
            .unstack("method_name"))
baseline["hard_f1"].style.background_gradient(axis=None, cmap="Greens")

method_name,index,index_enriched,json,json_constrained,json_occurrence,json_occurrence_constrained,json_occurrence_structured,json_occurrence_structured_constrained,json_structured,json_structured_constrained,xml
model,,,,,,,,,,,
NousResearch/Hermes-3-Llama-3.1-8B,0.165700,0.336700,0.673000,nan,0.632900,nan,0.607500,nan,0.630700,nan,0.580500
Qwen/Qwen3-8B,0.444600,0.561100,0.792900,0.751200,0.748700,0.732300,0.739000,0.713400,0.786300,0.740200,0.788100
google/gemma-4-31B-it,0.591500,0.878500,0.877800,0.892800,0.886400,0.899800,0.883300,0.893900,0.859700,0.867800,0.892200
gpt-5-mini,0.382800,0.789200,0.789900,nan,0.794700,nan,0.761100,nan,0.772400,nan,0.743400
meta-llama/Llama-3.3-70B-Instruct,0.277200,0.700100,0.817300,0.797900,0.815500,0.791300,0.815200,0.792700,0.819900,0.798700,0.828300
mistralai/Mistral-7B-Instruct-v0.3,0.132700,0.265000,0.573900,nan,0.558600,nan,0.527100,nan,0.513300,nan,0.449900
mistralai/Mistral-Small-24B-Instruct-2501,0.271500,0.685100,0.790700,0.762600,0.783500,0.752500,0.761100,0.730300,0.762500,0.738100,0.741400


In [14]:
# Our head-to-head row: their tagging method (xml) on NER, per model.
# This is the number our trie-constrained tagging has to beat.
xml_ner = (ner_results[ner_results["method_name"] == "xml"]
           .groupby("model")[["hard_f1", "soft_f1", "avg_completion_tokens"]]
           .agg(["mean", "std"])
           .round(4))
xml_ner

hard_f1         soft_f1          \
                                             mean     std    mean     std   
model                                                                       
NousResearch/Hermes-3-Llama-3.1-8B         0.5805  0.0832  0.6406  0.0724   
Qwen/Qwen3-8B                              0.7881  0.0872  0.8389  0.0747   
google/gemma-4-31B-it                      0.8922  0.0473  0.9282  0.0273   
gpt-5-mini                                 0.7434  0.0475  0.8026  0.0414   
meta-llama/Llama-3.3-70B-Instruct          0.8283  0.0692  0.8713  0.0620   
mistralai/Mistral-7B-Instruct-v0.3         0.4499  0.0799  0.5306  0.0689   
mistralai/Mistral-Small-24B-Instruct-2501  0.7414  0.0506  0.7941  0.0360   

                                          avg_completion_tokens            
                                                           mean       std  
model                                                                      
NousResearch/Hermes-3-Llama-3.1-8B                      61.5728   16.9868  
Qwen/Qwen3-8B                                          346.6553  302.4509  
google/gemma-4-31B-it                                  145.1470  217.8151  
gpt-5-mini                                              63.3279   15.5372  
meta-llama/Llama-3.3-70B-Instruct                       75.9501   23.2220  
mistralai/Mistral-7B-Instruct-v0.3                     101.2169   35.1522  
mistralai/Mistral-Small-24B-Instruct-2501               73.4375   26.6384

In [15]:
# Error breakdown -- their taxonomy of failure modes. `error_span_not_found` is the
# one constrained decoding is supposed to eliminate, so it is our headline diagnostic.
error_cols = [c for c in results.columns if c.startswith("error_")]
(ner_results
 .groupby("method_name")[error_cols]
 .mean()
 .round(3)
 .loc[:, lambda d: d.sum() > 0]
 .T)

method_name,index,index_enriched,json,json_constrained,json_occurrence,json_occurrence_constrained,json_occurrence_structured,json_occurrence_structured_constrained,json_structured,json_structured_constrained,xml
error_empty_response,6.939,3.056,0.003,0.033,0.003,0.018,0.003,0.020,0.005,0.025,0.033
error_format_error,1.725,3.318,0.551,0.081,0.579,0.126,0.060,0.048,0.051,0.053,13.863
error_span_not_found,0.000,0.000,6.282,0.000,5.591,0.000,4.522,0.000,5.152,0.000,0.000
error_partial_span_not_found,0.000,0.000,28.069,0.000,26.332,0.000,28.344,0.000,29.207,0.000,0.000
error_invalid_label,0.491,0.638,0.099,0.000,0.142,0.000,0.000,0.000,0.000,0.000,2.085
error_partial_invalid_label,11.318,19.772,3.107,0.000,3.836,0.000,0.000,0.000,0.000,0.000,11.748
error_empty_prediction,0.446,0.221,12.001,9.114,8.966,7.384,23.020,22.803,21.911,31.949,3.103
error_success,396.401,390.810,366.615,400.856,371.379,402.616,356.858,390.101,354.053,380.962,387.104
error_format_max_tokens_exceeded,0.579,0.072,0.005,5.402,0.070,5.826,0.102,4.823,0.076,3.846,0.008
error_format_unescaped_quotes,0.000,0.000,0.237,2.447,0.317,1.962,0.001,0.003,0.007,0.005,0.000


## 7. WMT24 error spans

Second usable task. Note `text` here is the **translation** and there is an extra
`source` field — a span labeling task over the output, conditioned on the source.
Labels are severities (`MINOR` / `MAJOR`), and spans are *not* token-aligned, so
this one needs true character-level handling rather than our BIO path.

In [16]:
ex = wmt["wmt-en-cs-news"][0]
print("source     :", ex["source"][:300])
print("translation:", ex["text"][:300])
print("languages  :", ex["source_language"], "->", ex["target_language"])
print("\nannotated:")
print(render(ex))

source     : Amid calls for his resignation over the abject failure of the SNP's NHS recovery plan, Matheson spoke of how the "heightened winter pressure" was "not unique to Scotland." Instead of "blame Westminster," the attempted defence this time was "Westminster's just as bad," as if that provided any comfort
translation: Vprostřed výzev k jeho rezignaci kvůli naprostému selhání plánu obnovy NHS od SNP Matheson hovořil o tom, jak „zvýšený zimní tlak“ nebyl „unikátní pro Skotsko“. Místo „obviňování Westminsteru“ byla obranná linie tentokrát „Westminster je stejně špatný“, jako by to byla nějaká útěcha pro legie pacien
languages  : en -> cs

annotated:
Vprostřed výzev k jeho rezignaci kvůli naprostému selhání plánu obnovy NHS od SNP Matheson hovořil o tom, jak „[zvýšený zimní tlak|MAJOR]“ nebyl „unikátní pro Skotsko“. Místo „obviňování Westminsteru“ byla obranná linie tentokrát „Westminster je stejně špatný“, jako by to byla nějaká útěcha pro legie pacientů nucených čekat nekonečné ho

In [17]:
# How token-aligned are WMT spans? (Expect: much less than NER.)
aligned = total = 0
for examples in wmt.values():
    for e in examples:
        starts = {a for a, _ in build_token_char_spans(e["text"].split(" "))}
        ends = {b for _, b in build_token_char_spans(e["text"].split(" "))}
        for s in e["spans"]:
            total += 1
            aligned += s["start"] in starts and s["end"] in ends
print(f"WMT spans on token boundaries: {aligned:,}/{total:,} ({100 * aligned / total:.1f}%)")

WMT spans on token boundaries: 792/1,919 (41.3%)


## 8. Where this leaves us

**Use for the paper.** UniversalNER is the comparison benchmark: 7,523 examples,
reproduced exactly, PER/ORG/LOC, and it drops into the existing CoNLL pipeline by
swapping the label set and `tokens = text.split(" ")`. WMT24 is a good second task
if we want breadth, but its spans are not token-aligned, so the BIO path does not apply.

**Reporting.** Report *their* pooled character-overlap F1 (hard and soft) alongside
our seqeval number — the two are not interchangeable, and only the former is
comparable to their table. Aggregate by pooling counts, then computing F1 once.

**Watch out.**
- Their models are `Qwen3-8B`, `Mistral-Small-24B`, `Llama-3.3-70B`, `Gemma-4-31B`,
  `gpt-5-mini`. Only Qwen3-8B overlaps with our current registry — matching at least
  one model exactly makes the comparison much stronger.
- They ran 5 seeds (42–46) and one-shot prompts; our current NER script is `N_ITERS = 1`.
- Use UNER **v1** (what this script fetches, and what they used). UNER v2 changed
  the English EWT/PUD annotations, so v2 numbers would not line up with their table.
- Their `results.csv` skips non-`*_fixed` constrained runs — an earlier processor bug.
  The `constrained_fixed` configs are canonical.